# CI/CD for Machine Learning

## What is CI/CD for ML?
Continuous Integration and Continuous Delivery for ML extends traditional software CI/CD to handle the unique challenges of ML: data changes, model training, evaluation gates, and reproducibility.

## ML CI/CD vs Traditional CI/CD
| Aspect | Traditional | ML |
|--------|-------------|----|
| What changes | Code | Code + Data + Model |
| Tests | Unit, integration | Code tests + model quality tests + data validation |
| Build artifact | Binary/package | Trained model |
| Deployment | App deployment | Model serving |
| Monitoring | App metrics | Model performance + data drift |

## Tools
- **GitHub Actions** CI/CD workflows
- **CML** (Continuous Machine Learning) ML metrics in PRs
- **DVC** Data + pipeline versioning
- **Great Expectations** Data validation
- **pytest** Testing ML code

## GitHub Actions for ML

### Basic ML Training Workflow
```yaml
# .github/workflows/train.yml
name: ML Training Pipeline

on:
  push:
    branches: [main]
    paths:
      - 'src/**'
      - 'params.yaml'
      - 'data/**'
  pull_request:
    branches: [main]
  schedule:
    - cron: '0 2 * * 1'  # Weekly Monday 2 AM

jobs:
  train:
    runs-on: ubuntu-latest
    
    steps:
    - name: Checkout repo
      uses: actions/checkout@v4

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.11'
        cache: 'pip'

    - name: Install dependencies
      run: pip install -r requirements.txt

    - name: Configure DVC
      run: |
        dvc remote modify myremote --local access_key_id ${{ secrets.AWS_ACCESS_KEY_ID }}
        dvc remote modify myremote --local secret_access_key ${{ secrets.AWS_SECRET_ACCESS_KEY }}

    - name: Pull data
      run: dvc pull

    - name: Run data validation
      run: python src/validate_data.py

    - name: Run tests
      run: pytest tests/ -v --cov=src --cov-report=xml

    - name: Upload coverage
      uses: codecov/codecov-action@v3

    - name: Train model
      run: dvc repro

    - name: Evaluate model
      run: python src/evaluate.py

    - name: Report metrics with CML
      env:
        REPO_TOKEN: ${{ secrets.GITHUB_TOKEN }}
      run: |
        echo '## ML Metrics' > report.md
        echo '```json' >> report.md
        cat metrics/scores.json >> report.md
        echo '```' >> report.md
        cml comment create report.md

    - name: Push to DVC
      if: github.ref == 'refs/heads/main'
      run: dvc push
```

In [1]:
# pytest tests for ML code
TEST_FILE = '''
import pytest
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- Data validation tests ---
class TestDataQuality:
    def test_no_missing_values(self):
        data = load_iris()
        df = pd.DataFrame(data.data, columns=data.feature_names)
        assert df.isnull().sum().sum() == 0, "Dataset has missing values"

    def test_correct_feature_count(self):
        data = load_iris()
        assert data.data.shape[1] == 4, f"Expected 4 features, got {data.data.shape[1]}"

    def test_target_classes(self):
        data = load_iris()
        unique_classes = np.unique(data.target)
        assert len(unique_classes) == 3, f"Expected 3 classes, got {len(unique_classes)}"

    def test_feature_ranges(self):
        data = load_iris()
        df = pd.DataFrame(data.data, columns=data.feature_names)
        for col in df.columns:
            assert df[col].min() >= 0, f"Negative values in {col}"
            assert df[col].max() < 100, f"Unreasonably large values in {col}"

# --- Model tests ---
@pytest.fixture
def trained_model():
    data = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        data.data, data.target, test_size=0.2, random_state=42
    )
    model = RandomForestClassifier(n_estimators=10, random_state=42)
    model.fit(X_train, y_train)
    return model, X_test, y_test

class TestModel:
    def test_model_accuracy_threshold(self, trained_model):
        model, X_test, y_test = trained_model
        accuracy = accuracy_score(y_test, model.predict(X_test))
        assert accuracy >= 0.90, f"Model accuracy {accuracy:.4f} below threshold 0.90"

    def test_model_output_shape(self, trained_model):
        model, X_test, _ = trained_model
        predictions = model.predict(X_test)
        assert predictions.shape == (len(X_test),)

    def test_model_output_classes(self, trained_model):
        model, X_test, _ = trained_model
        predictions = model.predict(X_test)
        assert set(predictions).issubset({0, 1, 2})

    def test_prediction_proba_sums_to_one(self, trained_model):
        model, X_test, _ = trained_model
        probas = model.predict_proba(X_test)
        np.testing.assert_allclose(probas.sum(axis=1), 1.0, atol=1e-6)

    def test_no_data_leakage(self):
        """Ensure test data was not used in training."""
        data = load_iris()
        X_train, X_test, y_train, y_test = train_test_split(
            data.data, data.target, test_size=0.2, random_state=42
        )
        # Check no overlap
        train_set = set(map(tuple, X_train))
        test_set = set(map(tuple, X_test))
        assert len(train_set & test_set) == 0, "Data leakage: train/test overlap"
'''
print("pytest tests written")
print(TEST_FILE[:300] + '...')

pytest tests written

import pytest
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- Data validation tests ---
class TestDataQuality:
   ...


## CML Continuous Machine Learning

CML reports ML metrics directly in GitHub/GitLab PR comments.

```bash
# Install CML
npm install -g @dvcorg/cml

# Create a report
python evaluate.py > metrics.txt

# Add confusion matrix plot
python plot_confusion.py

# Post comment to PR
echo '## Model Performance' > report.md
echo '```' >> report.md
cat metrics.txt >> report.md
echo '```' >> report.md
cml-publish confusion_matrix.png --md >> report.md
cml-send-comment report.md
```

## Model Validation Gate

```yaml
# .github/workflows/model_gate.yml
- name: Model Quality Gate
  run: |
    python << 'EOF'
    import json, sys
    with open('metrics/scores.json') as f:
        metrics = json.load(f)
    
    THRESHOLDS = {'accuracy': 0.90, 'f1_macro': 0.88}
    failed = []
    for metric, threshold in THRESHOLDS.items():
        if metrics.get(metric, 0) < threshold:
            failed.append(f"{metric}: {metrics[metric]:.4f} < {threshold}")
    
    if failed:
        print(f"QUALITY GATE FAILED: {failed}")
        sys.exit(1)
    print("Quality gate passed!")
    EOF
```

## Data Validation with Great Expectations

```python
import great_expectations as gx

context = gx.get_context()
batch = context.sources.pandas_default.read_csv('data/train.csv')

# Define expectations
batch.expect_column_values_to_not_be_null('feature1')
batch.expect_column_values_to_be_between('age', min_value=0, max_value=120)
batch.expect_column_values_to_be_in_set('label', value_set=[0, 1])
batch.expect_table_row_count_to_be_between(min_value=1000, max_value=1000000)
batch.expect_column_mean_to_be_between('income', min_value=20000, max_value=200000)

# Run validation
result = batch.validate()
if not result.success:
    raise ValueError(f"Data validation failed: {result}")
```

## Automated Retraining

```yaml
# .github/workflows/retrain.yml
name: Automated Retraining
on:
  schedule:
    - cron: '0 3 * * 0'  # Every Sunday 3 AM
  workflow_dispatch:     # Manual trigger
    inputs:
      force_retrain:
        description: 'Force retrain even if no drift'
        default: 'false'

jobs:
  check-drift:
    runs-on: ubuntu-latest
    outputs:
      should_retrain: ${{ steps.drift.outputs.drift_detected }}
    steps:
      - uses: actions/checkout@v4
      - name: Check data drift
        id: drift
        run: |
          python src/check_drift.py
          echo "drift_detected=$(cat drift_result.txt)" >> $GITHUB_OUTPUT

  retrain:
    needs: check-drift
    if: needs.check-drift.outputs.should_retrain == 'true'
    runs-on: ubuntu-latest
    steps:
      - name: Retrain model
        run: python src/train.py
```

## Additional Learning Resources

### CI/CD for ML
- [GitHub Actions Docs](https://docs.github.com/en/actions)
- [CML Docs](https://cml.dev/doc)
- [MLOps Zoomcamp CI/CD module](https://github.com/DataTalksClub/mlops-zoomcamp)

### Data Validation
- [Great Expectations Docs](https://docs.greatexpectations.io/)
- [Pandera](https://pandera.readthedocs.io/) DataFrame schema validation

### Testing ML
- [Made With ML Testing](https://madewithml.com/courses/mlops/testing/)
- [pytest docs](https://docs.pytest.org/)

### Papers
- [Towards ML Engineering: A Survey](https://arxiv.org/abs/2101.00925)